# پارسی‌گو (pocket) روی Colab — GPU/CPU هوشمند

همان موتور ONNX پروژه، بدون torch: کد از GitHub clone می‌شود، وزن‌ها از Hugging Face دانلود می‌شوند و موتور خودش بین CUDA و CPU بنچمارک می‌گیرد و بردنده را اجرا می‌کند.

**قبل از اجرا:** منو `Runtime → Change runtime type → T4 GPU` (یا هر GPU دیگر).

- وزن‌ها: [`Nimaone/pocket-tts-farsi-v2-onnx`](https://huggingface.co/Nimaone/pocket-tts-farsi-v2-onnx)
- کد: [`nimaone/persian_tts`](https://github.com/nimaone/persian_tts) (main)


In [ ]:
# ---------- تنظیمات ----------
GITHUB_REPO = "https://github.com/nimaone/persian_tts.git"
HF_REPO     = "Nimaone/pocket-tts-farsi-v2-onnx"
PARSIGO_EP  = "auto"    # auto | cpu | cuda | dml
TEXT        = "سلام! امروز هوا ابری است اما باران نمی‌بارد. فردا هم همین‌طور است."
VOICE       = "/content/parsigo_tts/voices/male_hello.wav"
SEED        = 42

In [ ]:
!nvidia-smi -L || echo "⚠️ GPU فعال نیست — Runtime > Change runtime type > GPU"

In [ ]:
# onnxruntime-gpu جایگزین نسخهٔ CPU کولب می‌شود؛ CPU provider داخلش باقی است
!pip install -q onnxruntime-gpu soundfile librosa pedalboard sentencepiece scipy
import onnxruntime as ort
print("ort", ort.__version__, "| providers:", ort.get_available_providers())

In [ ]:
import os, sys
if not os.path.isdir("/content/parsigo_tts"):
    !git clone --depth 1 {GITHUB_REPO} /content/parsigo_tts
else:
    !git -C /content/parsigo_tts pull --ff-only || true
sys.path.insert(0, "/content/parsigo_tts/scripts")
os.environ["PARSIGO_EP"] = PARSIGO_EP
!git -C /content/parsigo_tts log --oneline -1

In [ ]:
from huggingface_hub import snapshot_download, hf_hub_download
import os, shutil

# ۱) پکیج ONNX (از ریپوی خودت)
snapshot_download(repo_id=HF_REPO, local_dir="/content/parsigo_tts/model/onnx")
!ls /content/parsigo_tts/model/onnx | head -6

# ۲) دو فایل کوچک runtime از ریپوی عمومی مدل اصلی (توکنایزر + نرمال‌ساز)
os.makedirs("/content/parsigo_tts/model/v2", exist_ok=True)
for f in ("tokenizer_ph.model", "normalize_fa.py"):
    shutil.copy(hf_hub_download("mehdi-hf/pocket-tts-farsi-v2", f),
                f"/content/parsigo_tts/model/v2/{f}")
print("v2 files ready")

In [ ]:
from tts_onnx import OnnxTts

eng = OnnxTts(seed=SEED)      # اینجا بنچمارک CUDA مقابل CPU چاپ می‌شود
print("device:", eng.device_label, "| providers:", eng.prov)

In [ ]:
import time, soundfile as sf
from IPython.display import Audio, display

t0 = time.perf_counter()
wav = eng.synthesize_text(TEXT, VOICE, seed=SEED)
dt = time.perf_counter() - t0
sf.write("parsigo_pocket.wav", wav, eng.sample_rate)
print(f"{len(wav)/eng.sample_rate:.2f}s audio in {dt:.1f}s")
display(Audio(wav, rate=eng.sample_rate))

## نکته‌ها

- **انتخاب هوشمند:** موتور موقع بارگذاری یک گام را روی GPU و CPU بنچمارک می‌کند و بردنده را نگه می‌دارد؛ روی T4 باید `GPU (CUDA)` چاپ شود.
- **اجبار دستی:** `PARSIGO_EP = "cpu"` یا `"cuda"` در سلول تنظیمات و اجرای دوبارهٔ سلول‌ها.
- اگر providers فقط CPU بود: `pip install onnxruntime-gpu` را دوباره بزن و Runtime را restart کن.


## ⛔ هشدار جدی — مسئولیت کلون صدا با خودت است

سلول بعدی **صدای یک انسان را کلون می‌کند**. قبل از استفاده:

- فقط از صدای **خودت** یا با **اجازهٔ صریح صاحب صدا** استفاده کن.
- کلون صدای دیگران بدون رضایت (جعل هویت، دیزفیک، پیام فریبنده، اخبار جعلی) در اکثر کشورها **غیرقانونی** است و پیامدهای حقوقی دارد.
- مدل pocket با **CC-BY-NC-4.0** منتشر شده: استفادهٔ غیرتجاری.
- **هر چیزی که با این ابزار می‌سازی، مسئولیتش صددرصد با خودت است** — نه این کد، نه توسعه‌دهندگان آن. نرم‌افزار بدون هیچ ضمانتی ارائه می‌شود (AS-IS).


## 🎙️ تست سریع

فقط **همین سلول آخر** را ویرایش کن (متن و صدای مرجع را عوض/پاک کن) و اجرا بزن — سلول‌های بالا لازم نیست دوباره اجرا شوند.


In [ ]:
# ---------- تست سریع: فقط همین سلول را ویرایش و اجرا کن ----------
TEXT2  = "سلام! این یک تست سریع است."

# صدای مرجع برای کلون: یکی از صداهای آماده، یا مسیر فایل آپلودی خودت
VOICE2 = VOICE          # مثلاً "/content/parsigo_tts/voices/female_narration.wav"

import glob, os
print("صداهای آماده:", [os.path.basename(p) for p in
                        glob.glob("/content/parsigo_tts/voices/*.wav")])
# برای کلون با صدای خودت: فایل wav تمیز ۵-۱۰ ثانیه‌ای را در پنل Files کولاب
# بالا بگذار (یا خط زیر را از کامنت دربیاور و اجرا کن):
# from google.colab import files; VOICE2 = list(files.upload().keys())[0]

t0 = time.perf_counter()
wav2 = eng.synthesize_text(TEXT2, VOICE2, seed=SEED)
dt2 = time.perf_counter() - t0
sf.write("parsigo_quick.wav", wav2, eng.sample_rate)
print(f"{len(wav2)/eng.sample_rate:.2f}s audio in {dt2:.1f}s")
display(Audio(wav2, rate=eng.sample_rate))